# Kaggle

Фролова Анастасия Ивановна

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import random
from tqdm import tqdm
import zipfile
from google.colab import files
import shutil

# Монтируем Google Drive (опционально, если нужно сохранить модель)
from google.colab import drive
drive.mount('/content/drive')

# Устанавливаем seed для воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Проверяем наличие GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используется устройство: {device}')

print("="*50)
print("ШАГ 1: Загрузка и распаковка данных")
print("="*50)

# Сделать так:
if os.path.exists('/content/dataset.zip') or os.path.exists('/content/data.zip'):
    zip_file = '/content/dataset.zip' if os.path.exists('/content/dataset.zip') else '/content/data.zip'
    print(f"Найден архив: {zip_file}")
else:
    print("Пожалуйста, загрузите архив с данными (dataset.zip):")
    uploaded = files.upload()
    zip_file = list(uploaded.keys())[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Используется устройство: cuda
ШАГ 1: Загрузка и распаковка данных
Найден архив: /content/dataset.zip


In [ ]:
print("\nРаспаковка архива...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/')
print("Данные распакованы!")

# Проверяем структуру распакованных данных
print("\nСтруктура распакованных данных:")
print("Содержимое /content/:")
for item in os.listdir('/content/'):
    if 'dataset' in item.lower() or 'train' in item.lower() or 'test' in item.lower():
        print(f"  - {item}")

# Определяем пути к данным
# Проверяем различные возможные варианты структуры
possible_train_paths = [
    '/content/train_dataset/images/',
    '/content/train_dataset/train/images/',
    '/content/train/images/',
    '/content/train/',
]

possible_test_paths = [
    '/content/test_dataset/images/',
    '/content/test_dataset/test/images/',
    '/content/test/images/',
    '/content/test/',
]

train_images_dir = None
for path in possible_train_paths:
    if os.path.exists(path):
        train_images_dir = path
        break

test_images_dir = None
for path in possible_test_paths:
    if os.path.exists(path):
        test_images_dir = path
        break

if not train_images_dir or not test_images_dir:
    print("\nОшибка: не удалось найти папки с изображениями!")
    print("Содержимое /content/:")
    for root, dirs, files in os.walk('/content/'):
        level = root.replace('/content/', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            for d in dirs[:5]:
                print(f'{indent}  {d}/')
else:
    print(f"\nНайдены папки:")
    print(f"Train images: {train_images_dir}")
    print(f"Test images: {test_images_dir}")

# Ищем файл с метками для тренировочных данных
train_csv_path = None
possible_csv_paths = [
    '/content/train_dataset/train.csv',
    '/content/train_dataset/labels.csv',
    '/content/train/labels.csv',
    '/content/train.csv',
]

for path in possible_csv_paths:
    if os.path.exists(path):
        train_csv_path = path
        break

# Словарь для классов
emotions = {
    0: 'Anger',
    1: 'Contempt',
    2: 'Disgust',
    3: 'Fear',
    4: 'Happy',
    5: 'Neutral',
    6: 'Sad',
    7: 'Surprise'
}
num_classes = 8


Распаковка архива...
Данные распакованы!

Структура распакованных данных:
Содержимое /content/:
  - test_dataset
  - train_dataset
  - dataset.zip

Найдены папки:
Train images: /content/train_dataset/train/images/
Test images: /content/test_dataset/test/images/


In [68]:
class SELayer(nn.Module):
    """
    Squeeze-and-Excitation блок
    """
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

In [ ]:
# Расширенные аугментации для тренировки
train_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Трансформации для валидации и теста
val_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

class FacialExpressionDataset(Dataset):
    def __init__(self, images_dir, dataframe, transform=None, image_col='image_name', label_col='image_label'):
        """
        images_dir: папка с изображениями
        dataframe: pandas DataFrame с колонками [image_col, label_col]
        transform: трансформации для изображений
        image_col: название колонки с именами файлов
        label_col: название колонки с метками
        """
        self.images_dir = images_dir
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.image_col = image_col
        self.label_col = label_col

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Получаем имя файла и метку
        img_name = self.dataframe.iloc[idx][self.image_col]
        label = self.dataframe.iloc[idx][self.label_col]

        # Загружаем изображение
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        # Применяем трансформации
        if self.transform:
            image = self.transform(image)

        return image, label

def load_and_split_data(images_dir, csv_path, val_size=0.2, random_state=42):
    """
    Загружает CSV и разделяет данные на train/val
    """
    # Загружаем CSV
    df = pd.read_csv(csv_path)

    # Определяем названия колонок
    if 'filename' in df.columns:
        image_col = 'filename'
        label_col = 'label'
    elif 'ID' in df.columns:
        image_col = 'ID'
        label_col = 'TARGET'
    else:
        image_col = df.columns[0]
        label_col = df.columns[1]

    print(f"Использую колонки: {image_col} -> {label_col}")

    # Разделяем на train/val
    train_df, val_df = train_test_split(
        df,
        test_size=val_size,
        random_state=random_state,
        stratify=df[label_col]
    )

    print(f"\nРазделение данных:")
    print(f"Train: {len(train_df)} изображений")
    print(f"Val: {len(val_df)} изображений")

    return train_df, val_df, image_col, label_col

In [ ]:
class FastPowerfulCNN(nn.Module):
    """
    Упрощенная быстрая CNN
    """
    def __init__(self, num_classes=8):
        super(FastPowerfulCNN, self).__init__()

        # 96x96 -> 48x48
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # 48x48 -> 24x24
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # 24x24 -> 12x12
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # 12x12 -> 6x6
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # Global Average Pooling вместо Flatten
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Классификатор
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [71]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=30):
    """Функция обучения модели"""
    best_val_acc = 0
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            train_bar.set_postfix({
                'loss': running_loss / (train_bar.n + 1),
                'acc': 100 * correct / total
            })

        epoch_train_loss = running_loss / len(train_loader)
        epoch_train_acc = 100 * correct / total
        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]')
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                val_bar.set_postfix({
                    'loss': val_loss / (val_bar.n + 1),
                    'acc': 100 * correct / total
                })

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100 * correct / total
        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc)

        # Снижаем learning rate
        scheduler.step()

        print(f'Epoch {epoch+1}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

        # Сохраняем лучшую модель
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), 'best_model.pt')
            print(f'  -> Сохранена лучшая модель с Val Acc: {best_val_acc:.2f}%')

    return train_losses, val_losses, train_accs, val_accs

def generate_submission(test_images_dir, model, transform, output_file='submission.csv'):
    """Генерация файла сабмишна"""
    model.eval()
    results = {'ID': [], 'Target': []}

    # Получаем список изображений
    test_images = sorted([f for f in os.listdir(test_images_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    print(f"\nГенерация предсказаний для {len(test_images)} изображений...")

    with torch.no_grad():
        for img_name in tqdm(test_images):
            img_path = os.path.join(test_images_dir, img_name)
            image = Image.open(img_path).convert('RGB')
            image = transform(image).unsqueeze(0).to(device)

            outputs = model(image)
            _, predicted = torch.max(outputs.data, 1)

            results['ID'].append(img_name)
            results['Target'].append(predicted.item())

    # Создаем DataFrame и сохраняем
    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False)
    print(f"\nСабмишн файл сохранен как {output_file}")
    print(f"Первые 5 строк:")
    print(df.head())

    # Сохраняем в Google Drive
    drive_path = '/content/drive/MyDrive/submission.csv'
    df.to_csv(drive_path, index=False)
    print(f"\nФайл также сохранен в Google Drive: {drive_path}")

    return df

In [ ]:
print("\n" + "="*50)
print("ШАГ 5: Запуск обучения")
print("="*50)

# Пути
train_images_dir = "./train_dataset/train"
train_csv_path = "./train_dataset/train/train_dataset.csv"
test_images_dir = "./test_dataset/test"

# Загрузка и разделение данных
train_df, val_df, image_col, label_col = load_and_split_data(train_images_dir, train_csv_path)

# Определяем количество классов из данных
num_classes = len(train_df[label_col].unique())
print(f"\nКоличество классов: {num_classes}")

# Создание датасетов
train_dataset = FacialExpressionDataset(
    train_images_dir,
    train_df,
    transform=train_transform,
    image_col=image_col,
    label_col=label_col
)

val_dataset = FacialExpressionDataset(
    train_images_dir,
    val_df,
    transform=val_transform,
    image_col=image_col,
    label_col=label_col
)

# Создание DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# Инициализация модели
# model = FastPowerfulCNN(num_classes=num_classes, width_mult=0.75).to(device)
# Инициализация модели
model = FastPowerfulCNN(num_classes=num_classes).to(device)

# Подсчет параметров модели
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nМодель создана:")
print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")

# Определение функции потерь и оптимизатора
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Обучение модели
print("\nНачало обучения...")
train_losses, val_losses, train_accs, val_accs = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=30
)

# Загрузка лучшей модели для тестирования
model.load_state_dict(torch.load('best_model.pt'))
print(f"\nЛучшая модель загружена")


ШАГ 5: Запуск обучения
Использую колонки: image_name -> image_label

Разделение данных:
Train: 16167 изображений
Val: 4042 изображений

Количество классов: 8

Модель создана:
Всего параметров: 1,208,104
Обучаемых параметров: 1,208,104

Начало обучения...


Epoch 1/30 [Val]: 100%|██████████| 64/64 [00:05<00:00, 10.83it/s, loss=2.07, acc=16.5]


Epoch 1: Train Loss: 2.0741, Train Acc: 15.78% | Val Loss: 2.0670, Val Acc: 16.48%
  -> Сохранена лучшая модель с Val Acc: 16.48%


Epoch 2/30 [Val]: 100%|██████████| 64/64 [00:05<00:00, 10.78it/s, loss=2.06, acc=16.6]


Epoch 2: Train Loss: 2.0627, Train Acc: 16.45% | Val Loss: 2.0647, Val Acc: 16.60%
  -> Сохранена лучшая модель с Val Acc: 16.60%


...

Epoch 30: Train Loss: 0.7491, Train Acc: 72.07% | Val Loss: 0.7891, Val Acc: 71.47%
  -> Сохранена лучшая модель с Val Acc: 71.47%

Лучшая модель загружена

Генерация предсказаний для 0 изображений...


0it [00:00, ?it/s]


Сабмишн файл сохранен как submission.csv
Первые 5 строк:
Empty DataFrame
Columns: [ID, Target]
Index: []

Файл также сохранен в Google Drive: /content/drive/MyDrive/submission.csv

ОБУЧЕНИЕ ЗАВЕРШЕНО!
Файл submission.csv готов к загрузке на Kaggle
Файл находится в:
  - /content/submission.csv
  - /content/drive/MyDrive/submission.csv (если смонтирован Drive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("\n" + "="*50)
print("ПРОВЕРКА ТЕСТОВЫХ ПУТЕЙ")
print("="*50)

# Тестовые изображения в корне
test_images_dir = "./test"

# Проверяем что есть в корне
print("Содержимое корневой папки:")
for item in os.listdir("."):
    item_path = os.path.join(".", item)
    if os.path.isdir(item_path):
        print(f"  📁 {item}/")
        # Показываем содержимое первых папок
        try:
            subitems = os.listdir(item_path)[:5]
            for sub in subitems:
                print(f"     📄 {sub}")
        except:
            pass
    else:
        if item.endswith(('.csv', '.png', '.jpg')):
            print(f"  📄 {item}")

# Ищем папку с тестовыми изображениями
possible_test_paths = [
    "./test",          
    "./test_dataset",   
    "./test/images",   
    "./test_dataset/test",  
    "./test_dataset/test/images", 
]

test_images_dir = None
for path in possible_test_paths:
    if os.path.exists(path):
        # Проверяем есть ли изображения
        files = os.listdir(path)
        if any(f.endswith(('.png', '.jpg', '.jpeg')) for f in files):
            test_images_dir = path
            print(f"\n✅ Найдена папка с тестовыми изображениями: {test_images_dir}")
            print(f"   Найдено файлов: {len(files)}")
            print(f"   Примеры: {files[:3]}")
            break

if test_images_dir is None:
    print("\n❌ Не найдена папка с тестовыми изображениями!")
    print("Создайте папку 'test' в корне и положите туда изображения для предсказания")
    exit()

# Теперь передаем правильный путь в generate_submission
generate_submission(test_images_dir, model, val_transform, 'submission.csv')
from google.colab import files
files.download('submission.csv')


ПРОВЕРКА ТЕСТОВЫХ ПУТЕЙ
Содержимое корневой папки:
  📁 .config/
     📄 hidden_gcloud_config_universe_descriptor_data_cache_configs.db
     📄 config_sentinel
     📄 .last_survey_prompt.yaml
     📄 default_configs.db
     📄 .last_opt_in_prompt.yaml
  📁 test_dataset/
     📄 test
  📄 submission.csv
  📁 drive/
     📄 .shortcut-targets-by-id
     📄 MyDrive
     📄 .Trash-0
     📄 .Encrypted
  📁 train_dataset/
     📄 train
  📁 sample_data/
     📄 anscombe.json
     📄 README.md
     📄 california_housing_train.csv
     📄 mnist_test.csv
     📄 california_housing_test.csv

✅ Найдена папка с тестовыми изображениями: ./test_dataset/test/images
   Найдено файлов: 5053
   Примеры: ['8934.png', '14723.png', '18557.png']

Генерация предсказаний для 5053 изображений...


100%|██████████| 5053/5053 [00:12<00:00, 402.11it/s]


Сабмишн файл сохранен как submission.csv
Первые 5 строк:
          ID  Target
0  10001.png       7
1  10007.png       0
2  10010.png       0
3  10012.png       3
4  10013.png       2

Файл также сохранен в Google Drive: /content/drive/MyDrive/submission.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>